# CS201L – Artificial Intelligence Laboratory
## Lab 9: Regression – Predicting CO Concentration
**Dataset:** AirQuality.csv

## 0. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

# Make plots look clean
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

SEED = 42

## 1. Load and Preprocess Data

In [ ]:
# AirQuality.csv uses ';' as separator and ',' as decimal (European format)
df = pd.read_csv('AirQuality.csv', sep=';', decimal=',')

# Drop the unnamed trailing columns that appear in this dataset
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Drop Date and Time columns (not features)
df.drop(columns=['Date', 'Time'], errors='ignore', inplace=True)

# The dataset uses -200 as a sentinel for missing values
df.replace(-200, np.nan, inplace=True)

# Rename target column for convenience (strip spaces)
df.columns = df.columns.str.strip()

# The target column is 'CO(GT)' in the raw file
TARGET = 'CO(GT)'

# Drop rows where the target is NaN
df.dropna(subset=[TARGET], inplace=True)

# Drop rows where any feature is NaN
df.dropna(inplace=True)

# Rename target to 'CO' for clarity (matches lab sheet)
df.rename(columns={TARGET: 'CO'}, inplace=True)

print(f"Dataset shape after cleaning: {df.shape}")
df.head()

## 2. Data Splitting (60% Train / 20% Val / 20% Test)

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train+val, 20% test
train_val_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED)

# Second split: from the 80%, take 75% → 60% total train, 25% → 20% total val
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=SEED)

print(f"Train size  : {len(train_df)}  ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val size    : {len(val_df)}  ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test size   : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)")

## 3. Correlation Analysis

In [ ]:
# Compute Pearson correlation of each feature with CO target (on training data only)
correlations = train_df.corr()['CO'].drop('CO')
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("Correlation of each feature with CO concentration (absolute value, descending):")
print(correlations_sorted.to_string())

# Best feature
x_best = correlations_sorted.idxmax()
print(f"\n>>> Feature with highest absolute correlation: '{x_best}'")
print(f"    Correlation value: {correlations[x_best]:.4f}")

In [ ]:
# Bar chart of correlations
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if feat != x_best else 'crimson' for feat in correlations_sorted.index]
ax.barh(correlations_sorted.index, correlations_sorted.values, color=colors)
ax.set_xlabel('Absolute Pearson Correlation with CO')
ax.set_title('Feature Correlations with Target Variable (CO Concentration)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('correlation_bar.png', dpi=150)
plt.show()
print(f"Best feature (red): {x_best}")

## 4. Prepare Feature Arrays Using x_best

In [ ]:
X_train = train_df[[x_best]]
y_train = train_df['CO']

X_val = val_df[[x_best]]
y_val = val_df['CO']

X_test = test_df[[x_best]]
y_test = test_df['CO']

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape  : {X_val.shape}")
print(f"X_test shape : {X_test.shape}")

## 5. Simple Linear Regression

In [ ]:
# Build and train the simple linear regression model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

print(f"Intercept  (b0): {lin_reg.intercept_:.4f}")
print(f"Coefficient(b1): {lin_reg.coef_[0]:.4f}")
print(f"\nFitted equation: CO = {lin_reg.intercept_:.4f} + {lin_reg.coef_[0]:.4f} * {x_best}")

In [ ]:
# Generate predictions
y_train_pred = lin_reg.predict(X_train)
y_val_pred   = lin_reg.predict(X_val)
y_test_pred  = lin_reg.predict(X_test)

# Compute RMSE
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_train = rmse(y_train, y_train_pred)
rmse_val   = rmse(y_val,   y_val_pred)
rmse_test  = rmse(y_test,  y_test_pred)

print(f"Linear Regression RMSE")
print(f"  Train      : {rmse_train:.4f}")
print(f"  Validation : {rmse_val:.4f}")
print(f"  Test       : {rmse_test:.4f}")

In [ ]:
# Plot best fit line on training data
x_line = np.linspace(X_train[x_best].min(), X_train[x_best].max(), 300).reshape(-1, 1)
y_line = lin_reg.predict(x_line)

fig, ax = plt.subplots()
ax.scatter(X_train[x_best], y_train, alpha=0.3, s=15, color='steelblue', label='Training Data')
ax.plot(x_line, y_line, color='crimson', linewidth=2, label='Best Fit Line')
ax.set_xlabel(x_best)
ax.set_ylabel('CO Concentration (mg/m³)')
ax.set_title('Simple Linear Regression – Best Fit Line on Training Data')
ax.legend()
plt.tight_layout()
plt.savefig('linear_regression_fit.png', dpi=150)
plt.show()

## 6. Polynomial Curve Fitting (degrees 2 to 6)

In [ ]:
degrees = [2, 3, 4, 5, 6]
train_rmses = []
val_rmses   = []
poly_models = {}  # store (poly_transformer, trained_model) for each degree

for p in degrees:
    poly = PolynomialFeatures(degree=p, include_bias=True)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly   = poly.transform(X_val)

    poly_reg = LinearRegression()
    poly_reg.fit(X_train_poly, y_train)

    tr = rmse(y_train, poly_reg.predict(X_train_poly))
    vr = rmse(y_val,   poly_reg.predict(X_val_poly))

    train_rmses.append(tr)
    val_rmses.append(vr)
    poly_models[p] = (poly, poly_reg)

    print(f"Degree {p}  →  Train RMSE: {tr:.4f}  |  Val RMSE: {vr:.4f}")

In [ ]:
# Plot Training RMSE and Validation RMSE vs Polynomial Degree
fig, ax = plt.subplots()
ax.plot(degrees, train_rmses, marker='o', color='steelblue', label='Train RMSE')
ax.plot(degrees, val_rmses,   marker='s', color='crimson',   label='Validation RMSE')

best_degree = degrees[np.argmin(val_rmses)]
ax.axvline(x=best_degree, linestyle='--', color='gray', alpha=0.7, label=f'Best Degree = {best_degree}')

ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('RMSE')
ax.set_title('Train vs Validation RMSE across Polynomial Degrees')
ax.set_xticks(degrees)
ax.legend()
plt.tight_layout()
plt.savefig('polynomial_rmse_curve.png', dpi=150)
plt.show()

print(f"\nBest polynomial degree (lowest Val RMSE): {best_degree}")
print(f"  Val RMSE at degree {best_degree}: {min(val_rmses):.4f}")

## 7. Evaluate Best Polynomial Model on Test Set

In [ ]:
best_poly, best_poly_reg = poly_models[best_degree]

X_test_poly = best_poly.transform(X_test)
y_test_poly_pred = best_poly_reg.predict(X_test_poly)

rmse_test_poly = rmse(y_test, y_test_poly_pred)
print(f"Best Polynomial Model (degree={best_degree})")
print(f"  Test RMSE: {rmse_test_poly:.4f}")

In [ ]:
# Plot best polynomial fit on training data
x_line_poly = np.linspace(X_train[x_best].min(), X_train[x_best].max(), 500).reshape(-1, 1)
x_line_poly_transformed = best_poly.transform(x_line_poly)
y_line_poly = best_poly_reg.predict(x_line_poly_transformed)

fig, ax = plt.subplots()
ax.scatter(X_train[x_best], y_train, alpha=0.3, s=15, color='steelblue', label='Training Data')
ax.plot(x_line_poly, y_line_poly, color='crimson', linewidth=2,
        label=f'Polynomial Fit (degree={best_degree})')
ax.set_xlabel(x_best)
ax.set_ylabel('CO Concentration (mg/m³)')
ax.set_title(f'Best Polynomial Regression Fit (degree={best_degree}) on Training Data')
ax.legend()
plt.tight_layout()
plt.savefig('polynomial_best_fit.png', dpi=150)
plt.show()

## 8. Final Summary

In [ ]:
print("=" * 55)
print("            FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"Best feature (x_best)          : {x_best}")
print(f"Correlation with CO            : {correlations[x_best]:.4f}")
print()
print("Simple Linear Regression:")
print(f"  Train RMSE                   : {rmse_train:.4f}")
print(f"  Validation RMSE              : {rmse_val:.4f}")
print(f"  Test RMSE                    : {rmse_test:.4f}")
print()
print("Polynomial Regression:")
for d, tr, vr in zip(degrees, train_rmses, val_rmses):
    marker = " <-- best" if d == best_degree else ""
    print(f"  Degree {d}: Train={tr:.4f}, Val={vr:.4f}{marker}")
print()
print(f"Best Polynomial Degree         : {best_degree}")
print(f"Best Polynomial Test RMSE      : {rmse_test_poly:.4f}")
print("=" * 55)